# Matrix-weighted localization in $SE(3)$

This notebook estimates a short sequence of three-dimensional poses from observations of known landmarks and relative-pose measurements. Each landmark observation has a full anisotropic information matrix: errors perpendicular to the observation ray are weighted more strongly than errors along the ray.

We construct the same deterministic, noiseless problem as [`MatrixWeightedLocalizationExample.cpp`](../../../examples/MatrixWeightedLocalizationExample.cpp), perturb every pose, and recover the ground truth with Gauss--Newton. The example uses GTSAM's `KnownLandmarkFactorPose3`, so it exercises the same C++ factor and analytic Jacobian as the compiled example.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/MatrixWeightedLocalizationExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import sys

sys.path.insert(
    0,
    "/Users/avinashsubramanian/Desktop/Chordal/"
    "wt_matrix_weighted_localization/build/python",
)

import gtsam
print(gtsam.__file__)

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gtsam
from gtsam.symbol_shorthand import X

## Matrix-weighted measurement model

Let $T_{k0} = (C_{k0}, t_k) \in SE(3)$ map a world-frame point into sensor frame $k$. For a known world-frame landmark $m_l^0$ and its measured sensor-frame position $\widetilde m_{kl}$, the residual is

$$e_{kl}(T_{k0}) = C_{k0}m_l^0 + t_k - \widetilde m_{kl}.$$

The corresponding factor cost is

$$\frac{1}{2}e_{kl}^{\mathsf T}W_{kl}e_{kl},$$

where $W_{kl}=\Sigma_{kl}^{-1}$ is a full information matrix. The landmark is known data, so each landmark observation is a unary factor on one pose. Known landmarks anchor the trajectory; no separate pose prior is needed.

For the controlled anisotropic model, let $u$ be the unit observation ray, $P_r=uu^{\mathsf T}$ the radial projector, and $P_l=I-uu^{\mathsf T}$ the lateral projector. We choose lateral standard deviation $\sigma_l$ and radial standard deviation $\sigma_r=\rho\sigma_l$, where $\rho$ is the anisotropicity. Then

$$\Sigma=\sigma_l^2P_l+\sigma_r^2P_r,$$

and therefore

$$W=\frac{1}{\sigma_l^2}I+\left(\frac{1}{\sigma_r^2}-\frac{1}{\sigma_l^2}\right)uu^{\mathsf T}.$$

Thus $\rho=\sigma_r/\sigma_l=\sqrt{\operatorname{cond}(\Sigma)}$. With $\rho=10$, radial noise has ten times the standard deviation and one hundred times the variance of lateral noise.

In [ ]:
num_poses = 3
lateral_sigma = 0.01
anisotropicity = 10.0
radial_sigma = anisotropicity * lateral_sigma
odometry_sigma = 0.1

step = gtsam.Pose3(
    gtsam.Rot3.RzRyRx(0.03, -0.05, 0.08), np.array([0.3, -0.1, 0.2])
)
ground_truth = [gtsam.Pose3()]
for _ in range(1, num_poses):
    ground_truth.append(ground_truth[-1].compose(step))

landmarks = [
    np.array([4.0, 1.0, 2.0]),
    np.array([-1.0, 3.0, 5.0]),
    np.array([2.0, -4.0, 3.0]),
    np.array([5.0, 2.0, -1.0]),
]

lateral_precision = 1.0 / lateral_sigma**2
radial_precision = 1.0 / radial_sigma**2
print(f"sigma_l={lateral_sigma:.3f} m, sigma_r={radial_sigma:.3f} m")
print(f"sqrt(cond(Sigma))={anisotropicity:.1f}")

## Build the factor graph

Every pose observes all four noncoplanar landmarks. Measurements are exact in this first example, but their information matrices remain anisotropic and affect the objective's curvature. Consecutive poses are connected by exact, isotropically weighted Frobenius relative-pose factors.

In [ ]:
graph = gtsam.NonlinearFactorGraph()
ground_truth_values = gtsam.Values()
first_information = None

for k, pose in enumerate(ground_truth):
    ground_truth_values.insert(X(k), pose)
    for landmark in landmarks:
        measurement = pose.transformFrom(landmark)
        ray = measurement / np.linalg.norm(measurement)
        information = (
            lateral_precision * np.eye(3)
            + (radial_precision - lateral_precision) * np.outer(ray, ray)
        )
        if first_information is None:
            first_information = information
        graph.add(
            gtsam.KnownLandmarkFactorPose3(
                X(k),
                landmark,
                measurement,
                gtsam.noiseModel.Gaussian.Information(information),
            )
        )

odometry_model = gtsam.noiseModel.Isotropic.Sigma(6, odometry_sigma)
for i in range(num_poses - 1):
    graph.add(
        gtsam.FrobeniusBetweenFactorPose3(
            X(i),
            X(i + 1),
            ground_truth[i].between(ground_truth[i + 1]),
            odometry_model,
        )
    )

covariance_eigenvalues = np.linalg.eigvalsh(np.linalg.inv(first_information))
print(f"factors: {graph.size()}")
print("covariance eigenvalues:", covariance_eigenvalues)
print("sqrt condition number:", np.sqrt(covariance_eigenvalues[-1] / covariance_eigenvalues[0]))

## Perturb and optimize

We retract a progressively larger tangent-space perturbation from each ground-truth pose. The initial assignment has nonzero error. Since all measurements are mutually consistent and the landmarks fix the reference frame, Gauss--Newton should return to the unique zero-error solution.

In [ ]:
perturbation = np.array([0.02, -0.015, 0.01, 0.08, -0.05, 0.06])
initial = gtsam.Values()
for i, pose in enumerate(ground_truth):
    initial.insert(X(i), pose.retract((i + 1) * perturbation))

parameters = gtsam.GaussNewtonParams()
parameters.setMaxIterations(100)
parameters.setRelativeErrorTol(1e-12)
result = gtsam.GaussNewtonOptimizer(graph, initial, parameters).optimize()

ground_truth_error = graph.error(ground_truth_values)
initial_error = graph.error(initial)
final_error = graph.error(result)
pose_errors = np.array([
    np.linalg.norm(ground_truth[i].localCoordinates(result.atPose3(X(i))))
    for i in range(num_poses)
])

print(f"ground-truth error: {ground_truth_error:.3e}")
print(f"initial error:      {initial_error:.3e}")
print(f"final error:        {final_error:.3e}")
print(f"mean pose error:    {pose_errors.mean():.3e}")
print(f"maximum pose error: {pose_errors.max():.3e}")

assert ground_truth_error < 1e-8
assert final_error < 1e-8
assert pose_errors.max() < 1e-8

In [ ]:
# T_k0 maps world points into sensor coordinates, so the world-frame
# sensor position is the translation of T_k0^{-1}.
true_positions = np.vstack([pose.inverse().translation() for pose in ground_truth])
initial_positions = np.vstack([initial.atPose3(X(i)).inverse().translation() for i in range(num_poses)])
result_positions = np.vstack([result.atPose3(X(i)).inverse().translation() for i in range(num_poses)])
landmark_array = np.vstack(landmarks)

fig, axis = plt.subplots(figsize=(7, 5))
axis.plot(initial_positions[:, 0], initial_positions[:, 1], "o--", label="initial poses")
axis.plot(true_positions[:, 0], true_positions[:, 1], "ko-", label="ground truth")
axis.plot(result_positions[:, 0], result_positions[:, 1], "x-", label="optimized poses")
axis.scatter(landmark_array[:, 0], landmark_array[:, 1], marker="*", s=120, label="known landmarks")
axis.set(xlabel="world x [m]", ylabel="world y [m]", title="Matrix-weighted localization")
axis.axis("equal")
axis.grid(alpha=0.3)
axis.legend()
plt.tight_layout()

## Reading the result

The covariance eigenvalues should be $\sigma_l^2,\sigma_l^2,\sigma_r^2$, so their square-root condition number is the requested anisotropicity. The full information matrix is generally not diagonal in sensor coordinates because its radial eigenvector follows the observation ray.

The noiseless ground truth has zero objective, and Gauss--Newton returns the perturbed initialization to that solution. This is an algebraic validation of the matrix-weighted model. A statistical experiment would additionally sample measurement noise from each $\Sigma_{kl}$ and compare local and global solvers over multiple initializations and anisotropicity values.